# Regeneration notebook — two audit items

This notebook regenerates the **two quantities flagged in the deep audit** that were
not persisted to CSV, using the project's own `src.` modules (no reinvented pipeline):

1. **Per-class recovery recalls** (head-refit on the frozen prune80 model) → saves
   `explain/mitigate_per_class_recovery.csv`. This makes the per-class numbers in
   Figure 17 / the Results prose independently verifiable.
2. **Bootstrap 95% CIs on the headline per-class recall drops** (B=1000) → saves
   `explain/recall_drop_bootstrap_ci.csv`, so the headline collapse numbers in
   Table 2 / the prose can carry CIs.

Run top-to-bottom on the anchor seed. It reuses the saved anchor model and the exact
leakage-aware split, so the outputs are directly comparable to the rest of the paper.


In [2]:
# --- Colab bootstrap (config-driven; never hardcode a path) ---
try:
    from google.colab import drive; drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
except Exception:
    REPO = '.'
import os, sys
os.chdir(REPO); sys.path.insert(0, REPO)

import numpy as np, pandas as pd, torch
from src.config import CFG, PATHS, set_all_seeds
from src import data as D, models as M, compression as C, mitigate as MIT, metrics as MET, train as TR

SEED = CFG['anchor_seed']
set_all_seeds(SEED)
ARCH = 'cnn1d'; DATASET = 'ciciot2023'
# tables are written via PATHS.tables('explain', '<file>.csv')
print('repo:', REPO); print('tables dir:', PATHS.tables('explain','_probe').parent)


Mounted at /content/drive
repo: /content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression
tables dir: /content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression/results/tables/explain


In [3]:
# >>> RUNTIME CHECK — read this before the long cell <<<
# The fine-tune runs 8 epochs over 2.56M rows. On GPU this is ~10-20 min total;
# on CPU it can take 1-2+ HOURS. If the line below prints 'cpu', stop and switch:
#   Colab menu -> Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU
# then Runtime -> Restart and run all. No code changes needed (DEVICE auto-detects).
import torch
print('DEVICE:', 'cuda (GPU - good)' if torch.cuda.is_available() else 'cpu (SLOW - switch to T4 GPU)')
assert torch.cuda.is_available(), \
    'CPU runtime detected. Switch to a T4 GPU (Runtime > Change runtime type) before running the heavy cells.'


DEVICE: cuda (GPU - good)


## 1. Rebuild the anchor (M0) and the prune80 model from the frozen config
We reload the saved anchor checkpoint, rebuild its split/scaler/encoder exactly as trained, then reproduce prune80 with `compression.prune_and_finetune` (same gradient-masked fine-tune the pipeline used).

In [4]:
# Load the dataset + the EXACT primary split used everywhere in the paper
df = D.clean(D.load_raw(DATASET, subsample=True, seed=SEED), DATASET)
splits = D.temporal_within_capture_split(df, seed=SEED)
# use the SAME helper the models were trained with (avoids any feat-col drift)
feat_cols = TR.feature_columns(df)

# Rebuild label encoder + scaler the same way train.py does (TRAIN-only scaler fit)
from sklearn.preprocessing import LabelEncoder, StandardScaler
le = LabelEncoder().fit(df['label'].to_numpy())
scaler = StandardScaler().fit(df.loc[splits['train'], feat_cols].to_numpy(np.float32))
n_classes = len(le.classes_)
print('classes:', n_classes, '| train/val/test:',
      len(splits['train']), len(splits['val']), len(splits['test']))

# Load the saved anchor (M0) CNN checkpoint
ckpt_path = PATHS.model(DATASET, ARCH, 'M0', SEED)
# PyTorch >=2.6 defaults to weights_only=True, which rejects the NumPy
# scaler arrays saved in the checkpoint. This file is our own, so load full.
ck = torch.load(ckpt_path, map_location='cpu', weights_only=False)
# The config does not record channel widths and the training cells are stubs,
# so infer the architecture DIRECTLY from the checkpoint tensor shapes (always matches).
sd = ck['state_dict']
arch_kwargs = {}
if ARCH == 'cnn1d':
    ch1 = sd['conv.0.weight'].shape[0]          # e.g. 64
    ch2 = sd['conv.3.weight'].shape[0]          # e.g. 128
    arch_kwargs = {'channels': (int(ch1), int(ch2))}
    print('inferred CNN channels from checkpoint:', arch_kwargs['channels'])
elif ARCH == 'mlp':
    # body is Linear/ReLU/BN blocks; pull Linear out-features in order
    import re
    lin = sorted([k for k in sd if k.startswith('body.') and k.endswith('.weight') and sd[k].dim()==2],
                 key=lambda k: int(k.split('.')[1]))
    hidden = tuple(int(sd[k].shape[0]) for k in lin)
    arch_kwargs = {'hidden': hidden}
    print('inferred MLP hidden from checkpoint:', hidden)

anchor = M.build(ARCH, in_dim=len(feat_cols), n_classes=n_classes, **arch_kwargs)
anchor.load_state_dict(sd)
anchor = anchor.to(TR.DEVICE).eval()   # match model_p80's device (GPU) so forwards don't mismatch
print('loaded anchor:', ckpt_path)

# Reproduce prune80 (L1 unstructured to 80% + gradient-masked fine-tune)
set_all_seeds(SEED)
print('Fine-tuning prune80 (8 epochs over full train set; ~minutes on GPU)...')
model_p80, le_p, scaler_p = C.prune_and_finetune(
    anchor, df, DATASET, splits, SEED, amount=0.80, verbose=True)
model_p80.eval()
print('prune80 model rebuilt.')


classes: 34 | train/val/test: 2563172 549253 549271
inferred CNN channels from checkpoint: (64, 128)
loaded anchor: /content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression/models/ciciot2023/ciciot2023__cnn1d__M0__seed0.pt
Fine-tuning prune80 (8 epochs over full train set; ~minutes on GPU)...
    prune80 ft epoch 0
    prune80 ft epoch 1
    prune80 ft epoch 2
    prune80 ft epoch 3
    prune80 ft epoch 4
    prune80 ft epoch 5
    prune80 ft epoch 6
    prune80 ft epoch 7
prune80 model rebuilt.


## 2. Item A — per-class recovery recalls (head-refit on frozen prune80)
`mitigate.refit_head` re-fits ONLY a fresh linear head on the frozen compressed representation (body stays 80%-pruned), returning val/test logits. We then compute per-class test recall before (prune80) and after (head-refit), and save them.

In [5]:
# Per-class recall under prune80 (before recovery)
logits_p80_te, y_te = MIT._logits_for_split(model_p80, df, splits, scaler, feat_cols, le, 'test')
recall_p80, _ = MIT._per_class_recall(logits_p80_te, y_te)

# Head re-fit on frozen prune80 representation -> recovered logits
set_all_seeds(SEED)
Lva, yva, Lte, yte = MIT.refit_head(model_p80, df, splits, scaler, feat_cols, le,
                                    epochs=15, lr=1e-2, seed=SEED)
recall_refit, _ = MIT._per_class_recall(Lte, yte)

# Assemble per-class recovery table (all classes; mark measurable tier if available)
rows = []
for ci, cls in enumerate(le.classes_):
    rows.append({
        'label': cls,
        'recall_prune80': round(recall_p80.get(ci, float('nan')), 4),
        'recall_refit':   round(recall_refit.get(ci, float('nan')), 4),
        'recovery_gain':  round(recall_refit.get(ci, float('nan')) - recall_p80.get(ci, float('nan')), 4),
    })
rec_df = pd.DataFrame(rows).sort_values('recovery_gain', ascending=False)
out_a = PATHS.tables('explain', 'mitigate_per_class_recovery.csv')
rec_df.to_csv(out_a, index=False)
print('saved:', out_a)
# Show the classes cited in the paper
cited = ['DoS-UDP_Flood','DoS-HTTP_Flood','Recon-HostDiscovery','BenignTraffic',
         'DoS-SYN_Flood','Recon-OSScan','Recon-PortScan','MITM-ArpSpoofing']
print(rec_df[rec_df['label'].isin(cited)].to_string(index=False))


saved: /content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression/results/tables/explain/mitigate_per_class_recovery.csv
              label  recall_prune80  recall_refit  recovery_gain
      DoS-UDP_Flood          0.0000        0.7862         0.7862
     DoS-HTTP_Flood          0.2129        0.8331         0.6202
Recon-HostDiscovery          0.0759        0.6479         0.5719
      BenignTraffic          0.1785        0.6309         0.4524
      DoS-SYN_Flood          0.0000        0.1548         0.1548
   MITM-ArpSpoofing          0.0068        0.1327         0.1260
       Recon-OSScan          0.0001        0.1253         0.1253
     Recon-PortScan          0.0027        0.1183         0.1156


## 3. Item B — bootstrap 95% CIs on the headline per-class recall drops
For each measurable class we resample the test set (B=1000) and recompute the prune80 recall **drop** (M0 − prune80), reporting the percentile CI. We reuse `metrics.bootstrap_ci`. This attaches uncertainty to the recall drops in Table 2 / the prose.

In [6]:
# M0 per-class predictions on test (for the drop = M0 - prune80)
logits_M0_te, y_te2 = MIT._logits_for_split(anchor, df, splits, scaler, feat_cols, le, 'test')
pred_M0  = logits_M0_te.argmax(1)
pred_p80 = logits_p80_te.argmax(1)
assert (y_te2 == y_te).all(), 'label alignment check'

# measurable tier (from saved tiers file if present, else all)
try:
    tiers = pd.read_csv(PATHS.tables('baseline','cnn1d_M0_tiers.csv'), index_col=0)['final_tier']
    measurable = [c for c in le.classes_ if tiers.get(c) == 'measurable']
except Exception:
    measurable = list(le.classes_)

B = CFG['metrics']['bootstrap_B']
rng = np.random.default_rng(SEED)
rows = []
for cls in measurable:
    ci = int(le.transform([cls])[0])
    mask = (y_te == ci)
    idx = np.where(mask)[0]
    if len(idx) == 0:
        continue
    # bootstrap the DROP on this class's test samples
    drops = []
    corr_M0  = (pred_M0[idx]  == ci).astype(float)
    corr_p80 = (pred_p80[idx] == ci).astype(float)
    for _ in range(B):
        bs = rng.integers(0, len(idx), len(idx))
        drops.append(corr_M0[bs].mean() - corr_p80[bs].mean())
    drops = np.array(drops)
    rows.append({
        'label': cls,
        'recall_M0':    round(float(corr_M0.mean()), 4),
        'recall_p80':   round(float(corr_p80.mean()), 4),
        'drop':         round(float(corr_M0.mean() - corr_p80.mean()), 4),
        'drop_ci_lo':   round(float(np.percentile(drops, 2.5)), 4),
        'drop_ci_hi':   round(float(np.percentile(drops, 97.5)), 4),
        'support_test': int(len(idx)),
    })
ci_df = pd.DataFrame(rows).sort_values('drop', ascending=False)
out_b = PATHS.tables('explain', 'recall_drop_bootstrap_ci.csv')
ci_df.to_csv(out_b, index=False)
print('saved:', out_b)
print(ci_df.to_string(index=False))


saved: /content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression/results/tables/explain/recall_drop_bootstrap_ci.csv
               label  recall_M0  recall_p80    drop  drop_ci_lo  drop_ci_hi  support_test
       DoS-UDP_Flood     0.6823      0.0000  0.6823      0.6762      0.6886         22308
      DoS-HTTP_Flood     0.8669      0.2129  0.6540      0.6452      0.6631         10723
 Recon-HostDiscovery     0.7047      0.0759  0.6288      0.6220      0.6355         20030
       BenignTraffic     0.6791      0.1785  0.5006      0.4945      0.5068         29941
      Recon-PortScan     0.2467      0.0027  0.2440      0.2365      0.2512         12037
       DoS-SYN_Flood     0.1644      0.0000  0.1644      0.1594      0.1690         21674
        DNS_Spoofing     0.6128      0.4624  0.1504      0.1435      0.1569         26131
    MITM-ArpSpoofing     0.1251      0.0068  0.1183      0.1146      0.1220         27485
        Recon-OSScan     0.1145      0.0001  0.1145      0.1095    

## 4. End-of-unit discipline
Commit the two new CSVs so the audit values are persisted (run locally / in Colab):

```bash
cd $REPO && git add results/tables/explain/mitigate_per_class_recovery.csv \
    results/tables/explain/recall_drop_bootstrap_ci.csv \
    && git commit -m 'regen: per-class recovery recalls + bootstrap CIs on recall drops (audit items)' \
    && git push
```

Then download the two CSVs and hand them back so the paper's Table 13 / Figure 17 per-class
values and Table 2 recall-drop CIs can be wired in and re-audited.


## 5. Aggregate-metrics table (all 6 compression cells)

Computes the lean per-cell global metrics the paper uses to show that aggregate evaluation passes while
per-class trust collapses: **accuracy, macro-F1, ECE, model size, and sparsity**. Saves
`explain/global_metrics_by_cell.csv`. Uses the project's own `build_matrix` + `evaluate_cell`, so the
numbers match the rest of the paper. (Latency and weighted-F1 are intentionally omitted: latency cuts
against the paper's no-deployment-claims scoping, and weighted-F1/balanced-accuracy are redundant with
accuracy and macro-F1.)

In [7]:
# Build every compression cell with the project's own build_matrix, then evaluate each.
# LEAN metric set: accuracy, macro-F1, ECE, size, sparsity (the columns the paper actually uses).
import numpy as np, pandas as pd, torch
from sklearn.metrics import f1_score, accuracy_score
from src import metrics as MET

set_all_seeds(SEED)
matrix = C.build_matrix(anchor, df, DATASET, splits, SEED, arch=ARCH, verbose=True)

rows = []
for cell, entry in matrix.items():
    rec, mf1, probs, y_true, y_pred = C.evaluate_cell(entry, df, splits, le, scaler, feat_cols, which='test')
    if probs is None:
        print(f"  {cell:14s} (model unavailable; skipped)"); continue
    acc = accuracy_score(y_true, y_pred)
    ece = MET.overall_ece(probs, y_true, n_bins=15)
    sz  = entry["size"]                       # {"params","nonzero_params","sparsity"}
    bytes_per = 2 if entry["is_half"] else 1 if entry["is_int8"] else 4
    size_mb = sz.get("params", 0) * bytes_per / 1e6
    rows.append({'cell':cell, 'accuracy':round(acc,4), 'macro_f1':round(mf1,4),
                 'ece':round(float(ece),4), 'size_mb':round(size_mb,4),
                 'sparsity_pct':round(100*sz.get("sparsity",0),1)})
    print(f"  {cell:14s} acc={acc:.4f} macroF1={mf1:.4f} ece={float(ece):.4f} "
          f"size={size_mb:.3f}MB sparsity={100*sz.get('sparsity',0):.0f}%")

gm = pd.DataFrame(rows)
out_g = PATHS.tables('explain', 'global_metrics_by_cell.csv')
gm.to_csv(out_g, index=False)
print('\nsaved:', out_g)
print(gm.to_string(index=False))
print("\nKey check: accuracy should stay high (~0.95) across ALL cells incl prune80,")
print("while macro_f1 drops at prune80 and ece spikes at prune80 — that IS the paper's point.")


[M0]
   uncompressed anchor | {'params': 29730, 'nonzero_params': 29730, 'sparsity': 0.0}
[prune50]
    prune50 ft epoch 0
    prune50 ft epoch 1
    prune50 ft epoch 2
    prune50 ft epoch 3
    prune50 ft epoch 4
    prune50 ft epoch 5
    prune50 ft epoch 6
    prune50 ft epoch 7
   L1 prune 50% + finetune | {'params': 29730, 'nonzero_params': 15170, 'sparsity': 0.4897}
[prune80]
    prune80 ft epoch 0
    prune80 ft epoch 1
    prune80 ft epoch 2
    prune80 ft epoch 3
    prune80 ft epoch 4
    prune80 ft epoch 5
    prune80 ft epoch 6
    prune80 ft epoch 7
   L1 prune 80% + finetune | {'params': 29730, 'nonzero_params': 6433, 'sparsity': 0.7836}
[distillation]
    distill epoch 0
    distill epoch 1
    distill epoch 2
    distill epoch 3
    distill epoch 4
    distill epoch 5
    distill epoch 6
    distill epoch 7
    distill epoch 8
    distill epoch 9
    distill epoch 10
    distill epoch 11
   KD student (24,48) <- anchor | {'params': 5410, 'nonzero_params': 5410, 'sparsi